# GDS electrode capacitance with a buried ground plane

This notebook creates a new COMSOL 6.1 electrostatics model for the same GDS electrode layout used in `COMSOL_import_GDS.ipynb`, but adds an unpatterned, full-chip buried ground plane. The original notebook and model are not loaded or modified.

From top to bottom, the stack is: 0.5 um patterned top metal, 3 um SiO2, 0.5 um buried GND, 12 um SiO2, and 675 um silicon. The top metal remains centered at z=0, so the buried plane occupies -3.75 to -3.25 um.

The notebook configures geometry, named selections, materials, electrostatics, local mesh sizes, a stationary study, and capacitance evaluations. A dedicated mesh-only cell lets you test the mesh; no cell starts the stationary solve.

In [1]:
from pathlib import Path
import numpy as np
import mph
from jpype import JArray
from jpype.types import JBoolean, JInt

# Install MPh into this Jupyter kernel with `%pip install MPh` if needed.
client = mph.start(version='6.1')
print('COMSOL version:', client.version)
model = client.create('gds_buried_ground_capacitance')
j = model.java


COMSOL version: 6.1


## 1. Model configuration

All geometric and mesh lengths below are in micrometers. `RF_REFERENCE_XY_BBOX_UM` is the measured xy bounding box of RF domain 14 in the original geometry. The new geometry identifies RF by this physical footprint instead of assuming that entity numbering remains unchanged.

In [2]:
GDS_FILE = Path(r'C:\Users\Administrator\Downloads\COMSOL_MODEL_V3.gds')
GDS_CELL = 'UNNAMED_12'
ELECTRODE_LAYER = 'LAYER8'

TOP_METAL_THICKNESS_UM = 0.5
TOP_METAL_Z_MIN_UM = -0.25
TOP_METAL_Z_MAX_UM = +0.25
TOP_OXIDE_THICKNESS_UM = 3.0
GROUND_PLANE_THICKNESS_UM = 0.5
BOTTOM_OXIDE_THICKNESS_UM = 12.0
SILICON_THICKNESS_UM = 675.0

GROUND_PLANE_Z_MAX_UM = TOP_METAL_Z_MIN_UM - TOP_OXIDE_THICKNESS_UM
GROUND_PLANE_Z_MIN_UM = GROUND_PLANE_Z_MAX_UM - GROUND_PLANE_THICKNESS_UM
BOTTOM_OXIDE_Z_MAX_UM = GROUND_PLANE_Z_MIN_UM
BOTTOM_OXIDE_Z_MIN_UM = BOTTOM_OXIDE_Z_MAX_UM - BOTTOM_OXIDE_THICKNESS_UM
SILICON_Z_MAX_UM = BOTTOM_OXIDE_Z_MIN_UM
SILICON_Z_MIN_UM = SILICON_Z_MAX_UM - SILICON_THICKNESS_UM

CHIP_LENGTH_UM = 11976.0
CHIP_WIDTH_UM = 5846.51
# Split the enclosing air into sweepable layers beside thin stack features
# and coarse tetrahedral volumes above and below the chip.
AIR_GAP_Z_MIN_UM = TOP_METAL_Z_MIN_UM
AIR_GAP_Z_MAX_UM = TOP_METAL_Z_MAX_UM
AIR_Z_MIN_UM = -1000.0
AIR_Z_MAX_UM = 250.0
AIR_XY_PADDING_UM = 250.0

# Original RF domain 14 bounding box: xmin, xmax, ymin, ymax.
RF_REFERENCE_XY_BBOX_UM = np.array([
    9951.12988, 15276.0, -170.371994, 946.627991
])
RF_BBOX_TOLERANCE_UM = 1e-3
EXPECTED_TOP_METAL_DOMAIN_COUNT = 23

# Fine convergence mesh. Refinement is concentrated around RF, patterned
# metal, the 3 um oxide, and upper fringing fields. Proven settings are kept
# in the shielded lower stack to avoid reintroducing a meshing bottleneck.
MESH_RF = {'hmax': 20.0, 'hmin': 0.2, 'growth': 1.45}
MESH_PATTERNED_GND = {'hmax': 50.0, 'hmin': 0.2, 'growth': 1.60}
MESH_BURIED_GND = {'hmax': 150.0, 'hmin': 4.0, 'growth': 1.70}
MESH_SIO2_TOP = {'hmax': 50.0, 'hmin': 0.5, 'growth': 1.50}
MESH_SIO2_BOTTOM = {'hmax': 200.0, 'hmin': 4.0, 'growth': 1.70}
MESH_SILICON = {'hmax': 400.0, 'hmin': 20.0, 'growth': 1.75}
MESH_AIR_GAP = {'hmax': 50.0, 'hmin': 0.2, 'growth': 1.60}
MESH_AIR_SIDE = {'hmax': 200.0, 'hmin': 4.0, 'growth': 1.75}
MESH_AIR_UPPER = {'hmax': 600.0, 'hmin': 0.5, 'growth': 1.80}
MESH_AIR_LOWER = {'hmax': 800.0, 'hmin': 20.0, 'growth': 1.85}
MESH_RF_AIR = {'hmax': 50.0, 'hmin': 0.5, 'growth': 1.70}

if not GDS_FILE.is_file():
    raise FileNotFoundError(f'GDS file not found: {GDS_FILE}')
if GDS_FILE.suffix.lower() != '.gds':
    raise ValueError('The layout file must have a .gds extension.')
GDS_FILE = GDS_FILE.resolve()

print('GDS:', GDS_FILE)
print('z coordinates [um]:')
print('  top metal   :', TOP_METAL_Z_MIN_UM, TOP_METAL_Z_MAX_UM)
print('  top SiO2    :', GROUND_PLANE_Z_MAX_UM, TOP_METAL_Z_MIN_UM)
print('  buried GND  :', GROUND_PLANE_Z_MIN_UM, GROUND_PLANE_Z_MAX_UM)
print('  bottom SiO2 :', BOTTOM_OXIDE_Z_MIN_UM, BOTTOM_OXIDE_Z_MAX_UM)
print('  silicon     :', SILICON_Z_MIN_UM, SILICON_Z_MAX_UM)
assert np.isclose(TOP_METAL_Z_MIN_UM-GROUND_PLANE_Z_MAX_UM, 3.0)
assert np.isclose(GROUND_PLANE_Z_MAX_UM-GROUND_PLANE_Z_MIN_UM, 0.5)


GDS: C:\Users\Administrator\Downloads\COMSOL_MODEL_V3.gds
z coordinates [um]:
  top metal   : -0.25 0.25
  top SiO2    : -3.25 -0.25
  buried GND  : -3.75 -3.25
  bottom SiO2 : -15.75 -3.75
  silicon     : -690.75 -15.75


## 2. Import the patterned top electrode

Only `LAYER8` is imported. It is extruded from -0.25 to +0.25 um so the patterned metal remains a selectable set of 3D conductor domains.

In [3]:
j.component().create('comp1', JBoolean(True))
comp = j.component('comp1')
comp.geom().create('geom1', 3)
geom = comp.geom('geom1')

geom.create('imp1', 'Import')
imp = geom.feature('imp1')
imp.set('filename', str(GDS_FILE))
imp.set('updategeomunit', JBoolean(True))
imp.set('grouping', 'layer')
imp.set('importtype', 'full3d')
imp.set('manualelevation', 'on')
imp.set('intbnd', 'off')
imp.set('findarcs', 'auto')
imp.set('repairgeom', 'on')
imp.set('repairtoltype', 'auto')
imp.set('selresult', 'on')
imp.set('selresultshow', 'all')
imp.set('sellayer', 'on')
imp.set('sellayershow', 'all')

ecad_type = str(imp.getString('ecadtype')).lower()
if ecad_type != 'gds':
    raise RuntimeError(f'COMSOL identified ECAD type {ecad_type!r}, not GDS.')
layer_table = np.asarray(imp.getStringMatrix('layerprop'), dtype=object)
if layer_table.size == 0:
    raise RuntimeError('COMSOL found no GDS layers.')
if GDS_CELL:
    imp.set('cell', GDS_CELL)

layer_names = [str(row[0]).upper() for row in layer_table]
import_flags = ['on' if name == ELECTRODE_LAYER else 'off'
                for name in layer_names]
if not any(flag == 'on' for flag in import_flags):
    raise RuntimeError(f'{ELECTRODE_LAYER} is absent; layers={layer_names}')
imp.set('importlayer', import_flags)
imp.set('height', [
    f'{TOP_METAL_THICKNESS_UM}[um]' if flag == 'on' else '0[um]'
    for flag in import_flags
])
imp.set('elevation', [
    f'{TOP_METAL_Z_MIN_UM}[um]' if flag == 'on' else '0[um]'
    for flag in import_flags
])
try:
    imp.importData()
    geom.run('imp1')
except Exception as exc:
    raise RuntimeError(
        'GDS import failed. Check the ECAD Import Module license and file.'
    ) from exc

object_names = list(imp.objectNames())
if not object_names:
    raise RuntimeError('The GDS import produced no electrode objects.')
measure_import = geom.measure()
measure_import.selection().set(object_names)
electrode_bbox = np.asarray(measure_import.getBoundingBox(), dtype=float)
exmin, exmax, eymin, eymax, ezmin, ezmax = electrode_bbox
assert np.allclose([ezmin, ezmax],
                   [TOP_METAL_Z_MIN_UM, TOP_METAL_Z_MAX_UM])
print('Geometry unit:', geom.lengthUnit())
print('Top-metal bounding box [um]:', electrode_bbox)


Geometry unit: µm
Top-metal bounding box [um]: [ 5.27600000e+03  1.62760000e+04 -2.63892000e+03  3.20857007e+03
 -2.50000000e-01  2.50000000e-01]


## 3. Construct the dielectric stack, full-chip ground plane, and air

The chip right edge follows the imported layout right edge and the chip is centered on the layout in y. The buried plane uses exactly the chip length and width. Air encloses the complete chip with 250 um lateral padding, extending from z = -1000 um to z = 250 um. It is partitioned at every horizontal stack interface so thin regions beside the metal and dielectrics can be swept instead of tetrahedralized.

In [4]:
chip_xmax = exmax
chip_xmin = chip_xmax - CHIP_LENGTH_UM
chip_ycenter = 0.5 * (eymin + eymax)
chip_ymin = chip_ycenter - CHIP_WIDTH_UM/2
chip_ymax = chip_ymin + CHIP_WIDTH_UM

def add_block(tag, label, position_um, size_um):
    geom.create(tag, 'Block')
    block = geom.feature(tag)
    block.label(label)
    block.set('base', 'corner')
    block.set('pos', [f'{value:.12g}' for value in position_um])
    block.set('size', [f'{value:.12g}' for value in size_um])
    block.set('selresult', 'on')
    block.set('selresultshow', 'all')
    return block

add_block('oxide_top', '3 um top SiO2',
          [chip_xmin, chip_ymin, GROUND_PLANE_Z_MAX_UM],
          [CHIP_LENGTH_UM, CHIP_WIDTH_UM, TOP_OXIDE_THICKNESS_UM])
add_block('ground_plane', '0.5 um full-chip buried GND',
          [chip_xmin, chip_ymin, GROUND_PLANE_Z_MIN_UM],
          [CHIP_LENGTH_UM, CHIP_WIDTH_UM, GROUND_PLANE_THICKNESS_UM])
add_block('oxide_bottom', '12 um bottom SiO2',
          [chip_xmin, chip_ymin, BOTTOM_OXIDE_Z_MIN_UM],
          [CHIP_LENGTH_UM, CHIP_WIDTH_UM, BOTTOM_OXIDE_THICKNESS_UM])
add_block('silicon', '675 um silicon',
          [chip_xmin, chip_ymin, SILICON_Z_MIN_UM],
          [CHIP_LENGTH_UM, CHIP_WIDTH_UM, SILICON_THICKNESS_UM])

air_xmin = chip_xmin - AIR_XY_PADDING_UM
air_ymin = chip_ymin - AIR_XY_PADDING_UM
air_size_x = CHIP_LENGTH_UM + 2*AIR_XY_PADDING_UM
air_size_y = CHIP_WIDTH_UM + 2*AIR_XY_PADDING_UM

# A perforated 0.5 um slab surrounds the finite-thickness top electrodes.
# Since every electrode passes completely through it, this air domain can be
# swept through one layer instead of filled with thin tetrahedra.
add_block(
    'air_gap_box', '0.5 um sweepable near-electrode air slab',
    [air_xmin, air_ymin, AIR_GAP_Z_MIN_UM],
    [air_size_x, air_size_y, AIR_GAP_Z_MAX_UM-AIR_GAP_Z_MIN_UM],
)
geom.create('dif_air_gap', 'Difference')
dif_air_gap = geom.feature('dif_air_gap')
dif_air_gap.label('Near-electrode air slab minus patterned metal')
dif_air_gap.selection('input').set(['air_gap_box'])
dif_air_gap.selection('input2').set(['imp1'])
dif_air_gap.set('keepsubtract', 'on')
dif_air_gap.set('intbnd', 'on')
dif_air_gap.set('selresult', 'on')
dif_air_gap.set('selresultshow', 'all')

# The upper-air block starts above all metal sidewalls and therefore contains
# no submicrometer thickness that a tetrahedral mesher must resolve.
add_block(
    'air_upper', 'Coarse upper air',
    [air_xmin, air_ymin, AIR_GAP_Z_MAX_UM],
    [air_size_x, air_size_y, AIR_Z_MAX_UM-AIR_GAP_Z_MAX_UM],
)

# Rectangular air rings enclose every side of the solid chip stack.  Each
# ring is a simple outer slab minus the corresponding full-chip layer and
# is therefore sweepable without forcing tetrahedra through thin layers.
def add_air_ring(box_tag, difference_tag, label, z_min_um, z_max_um,
                 subtract_tag):
    add_block(
        box_tag, f'{label} outer slab',
        [air_xmin, air_ymin, z_min_um],
        [air_size_x, air_size_y, z_max_um-z_min_um],
    )
    geom.create(difference_tag, 'Difference')
    ring = geom.feature(difference_tag)
    ring.label(label)
    ring.selection('input').set([box_tag])
    ring.selection('input2').set([subtract_tag])
    ring.set('keepsubtract', 'on')
    ring.set('intbnd', 'on')
    ring.set('selresult', 'on')
    ring.set('selresultshow', 'all')
    return ring

add_air_ring(
    'air_side_top_box', 'dif_air_side_top',
    'Side air beside 3 um top SiO2',
    GROUND_PLANE_Z_MAX_UM, TOP_METAL_Z_MIN_UM, 'oxide_top',
)
add_air_ring(
    'air_side_plane_box', 'dif_air_side_plane',
    'Side air beside 0.5 um buried GND',
    GROUND_PLANE_Z_MIN_UM, GROUND_PLANE_Z_MAX_UM, 'ground_plane',
)
add_air_ring(
    'air_side_bottom_box', 'dif_air_side_bottom',
    'Side air beside 12 um bottom SiO2',
    BOTTOM_OXIDE_Z_MIN_UM, BOTTOM_OXIDE_Z_MAX_UM, 'oxide_bottom',
)
add_air_ring(
    'air_side_silicon_box', 'dif_air_side_silicon',
    'Side air beside 675 um silicon',
    SILICON_Z_MIN_UM, SILICON_Z_MAX_UM, 'silicon',
)

# A full lower-air block closes the enclosure beneath the silicon.
add_block(
    'air_lower', 'Coarse lower air',
    [air_xmin, air_ymin, AIR_Z_MIN_UM],
    [air_size_x, air_size_y, SILICON_Z_MIN_UM-AIR_Z_MIN_UM],
)
geom.run()

final_bbox = np.asarray(geom.getBoundingBox(), dtype=float)
expected_bbox = np.array([
    air_xmin, air_xmin+air_size_x, air_ymin, air_ymin+air_size_y,
    AIR_Z_MIN_UM, AIR_Z_MAX_UM,
])
if not np.allclose(final_bbox, expected_bbox):
    raise RuntimeError(
        f'Air enclosure bounds are incorrect: {final_bbox}; '
        f'expected {expected_bbox}'
    )
print('Chip x [um]:', chip_xmin, chip_xmax)
print('Chip y [um]:', chip_ymin, chip_ymax)
print('Full model bounding box [um]:', final_bbox)


Chip x [um]: 4300.000000000002 16276.000000000002
Chip y [um]: -2638.4299658203126 3208.0800341796876
Full model bounding box [um]: [ 4050.         16526.         -2888.42996582  3458.08003418
 -1000.           250.        ]


## 4. Create and validate domain and conductor selections

COMSOL can renumber domains after a geometry change. Therefore RF is matched to the original RF conductor's measured xy footprint. Every other imported top-metal domain plus the full buried plane is assigned to GND. Assertions require complete, nonoverlapping conductor coverage.

In [5]:
selection_tags = [str(tag) for tag in comp.selection().tags()]
top_domain_matches = [
    tag for tag in selection_tags
    if 'imp1' in tag and ELECTRODE_LAYER.lower() in tag.lower()
    and tag.lower().endswith('_dom')
]
if len(top_domain_matches) != 1:
    raise RuntimeError(
        f'Cannot uniquely find imported top-metal selection: {top_domain_matches}'
    )
top_metal_selection_tag = top_domain_matches[0]
top_metal_domain_ids = sorted(
    int(v) for v in comp.selection(top_metal_selection_tag).entities(JInt(3))
)
if len(top_metal_domain_ids) != EXPECTED_TOP_METAL_DOMAIN_COUNT:
    raise RuntimeError(
        f'Expected {EXPECTED_TOP_METAL_DOMAIN_COUNT} top-metal domains; '
        f'found {len(top_metal_domain_ids)}: {top_metal_domain_ids}'
    )

air_gap_ids = sorted(int(v) for v in
    comp.selection('geom1_dif_air_gap_dom').entities(JInt(3)))
air_upper_ids = sorted(int(v) for v in
    comp.selection('geom1_air_upper_dom').entities(JInt(3)))
air_side_top_ids = sorted(int(v) for v in
    comp.selection('geom1_dif_air_side_top_dom').entities(JInt(3)))
air_side_plane_ids = sorted(int(v) for v in
    comp.selection('geom1_dif_air_side_plane_dom').entities(JInt(3)))
air_side_bottom_ids = sorted(int(v) for v in
    comp.selection('geom1_dif_air_side_bottom_dom').entities(JInt(3)))
air_side_silicon_ids = sorted(int(v) for v in
    comp.selection('geom1_dif_air_side_silicon_dom').entities(JInt(3)))
air_lower_ids = sorted(int(v) for v in
    comp.selection('geom1_air_lower_dom').entities(JInt(3)))
air_ids = sorted(set(
    air_gap_ids + air_upper_ids + air_side_top_ids +
    air_side_plane_ids + air_side_bottom_ids +
    air_side_silicon_ids + air_lower_ids
))
top_oxide_ids = sorted(int(v) for v in
                       comp.selection('geom1_oxide_top_dom').entities(JInt(3)))
ground_plane_domain_ids = sorted(int(v) for v in
    comp.selection('geom1_ground_plane_dom').entities(JInt(3)))
bottom_oxide_ids = sorted(int(v) for v in
    comp.selection('geom1_oxide_bottom_dom').entities(JInt(3)))
silicon_ids = sorted(int(v) for v in
                     comp.selection('geom1_silicon_dom').entities(JInt(3)))
if len(ground_plane_domain_ids) != 1:
    raise RuntimeError(
        f'Buried plane should be one domain; got {ground_plane_domain_ids}'
    )

domain_measure = comp.measure()
domain_measure.selection().geom('geom1', JInt(3))
def domain_bbox(domain_id):
    domain_measure.selection().set(JArray(JInt)([int(domain_id)]))
    return np.asarray(domain_measure.getBoundingBox(), dtype=float)

rf_domain_ids = []
top_domain_bboxes = {}
for domain_id in top_metal_domain_ids:
    bbox = domain_bbox(domain_id)
    top_domain_bboxes[domain_id] = bbox
    if np.allclose(bbox[:4], RF_REFERENCE_XY_BBOX_UM, rtol=0.0,
                   atol=RF_BBOX_TOLERANCE_UM):
        rf_domain_ids.append(domain_id)
if len(rf_domain_ids) != 1:
    errors = {domain_id: float(np.max(np.abs(bbox[:4]-RF_REFERENCE_XY_BBOX_UM)))
              for domain_id, bbox in top_domain_bboxes.items()}
    raise RuntimeError(
        f'RF footprint match is not unique: {rf_domain_ids}. '
        f'Maximum bbox errors [um]: {errors}'
    )

top_gnd_domain_ids = sorted(set(top_metal_domain_ids)-set(rf_domain_ids))
gnd_domain_ids = sorted(set(top_gnd_domain_ids+ground_plane_domain_ids))
all_conductor_ids = sorted(set(top_metal_domain_ids+ground_plane_domain_ids))
if set(rf_domain_ids) & set(gnd_domain_ids):
    raise RuntimeError('RF and GND domain selections overlap.')
if set(rf_domain_ids) | set(gnd_domain_ids) != set(all_conductor_ids):
    raise RuntimeError('RF and GND do not cover all conductor domains.')

def adjacent_boundaries(domain_ids):
    result = set()
    for domain_id in domain_ids:
        result.update(int(v) for v in geom.getAdj(
            JInt(3), JInt(2), JInt(domain_id)
        ))
    return sorted(result)

rf_boundary_ids = adjacent_boundaries(rf_domain_ids)
patterned_gnd_boundary_ids = adjacent_boundaries(top_gnd_domain_ids)
buried_gnd_boundary_ids = adjacent_boundaries(ground_plane_domain_ids)
gnd_boundary_ids = sorted(set(patterned_gnd_boundary_ids +
                              buried_gnd_boundary_ids))
if set(rf_boundary_ids) & set(gnd_boundary_ids):
    raise RuntimeError('RF and GND boundary selections overlap.')
if set(patterned_gnd_boundary_ids) & set(buried_gnd_boundary_ids):
    raise RuntimeError('Patterned and buried GND boundaries overlap.')

def explicit_selection(tag, label, dimension, entity_ids):
    selection = comp.selection().create(tag, 'Explicit')
    selection.label(label)
    selection.geom('geom1', JInt(dimension))
    selection.set(JArray(JInt)(list(entity_ids)))
    return selection

sio2_ids = sorted(set(top_oxide_ids+bottom_oxide_ids))
dielectric_ids = sorted(set(air_ids+sio2_ids+silicon_ids))
dielectric_boundary_ids = set(adjacent_boundaries(dielectric_ids))
uncovered_conductor_boundaries = sorted(
    (set(rf_boundary_ids) | set(gnd_boundary_ids)) - dielectric_boundary_ids
)
if uncovered_conductor_boundaries:
    raise RuntimeError(
        'Some conductor faces do not touch a solved dielectric domain: '
        f'{uncovered_conductor_boundaries}. The air enclosure is incomplete.'
    )
explicit_selection('AIR_GAP', '01 Domains | Air near metal (swept)', 3,
                   air_gap_ids)
explicit_selection('AIR_SIDE_TOP', '01 Domains | Air beside top SiO2', 3,
                   air_side_top_ids)
explicit_selection('AIR_SIDE_PLANE', '01 Domains | Air beside buried GND', 3,
                   air_side_plane_ids)
explicit_selection('AIR_SIDE_BOTTOM',
                   '01 Domains | Air beside bottom SiO2', 3,
                   air_side_bottom_ids)
explicit_selection('AIR_SIDE_SILICON', '01 Domains | Air beside silicon', 3,
                   air_side_silicon_ids)
explicit_selection('AIR_LOWER', '01 Domains | Air below chip', 3,
                   air_lower_ids)
explicit_selection('AIR_UPPER', '01 Domains | Air upper (tetrahedral)', 3,
                   air_upper_ids)
explicit_selection('AIR', '01 Domains | All air', 3, air_ids)
explicit_selection('SIO2_TOP', '01 Domains | SiO2 top (3 um)', 3,
                   top_oxide_ids)
explicit_selection('SIO2_BOTTOM', '01 Domains | SiO2 bottom (12 um)', 3,
                   bottom_oxide_ids)
explicit_selection('SIO2', '01 Domains | All SiO2', 3, sio2_ids)
explicit_selection('SILICON', '01 Domains | Silicon', 3, silicon_ids)
explicit_selection('DIELECTRICS', '01 Domains | All solved dielectrics', 3,
                   dielectric_ids)
explicit_selection('RF', '02 Electrodes | RF boundaries', 2, rf_boundary_ids)
explicit_selection('GND_PATTERNED', '02 Electrodes | Patterned GND', 2,
                   patterned_gnd_boundary_ids)
explicit_selection('GND_BURIED', '02 Electrodes | Buried GND plane', 2,
                   buried_gnd_boundary_ids)
explicit_selection('GND', '02 Electrodes | All GND boundaries', 2,
                   gnd_boundary_ids)

# Identify the horizontal source and target faces for each swept domain.
# Restricting candidates to boundaries adjacent to the requested domain avoids
# accidentally selecting a different coplanar face elsewhere in the model.
boundary_measure = comp.measure()
boundary_measure.selection().geom('geom1', JInt(2))
def horizontal_faces(domain_ids, z_um, tolerance_um=1e-6):
    matches = []
    for boundary_id in adjacent_boundaries(domain_ids):
        boundary_measure.selection().set(JArray(JInt)([boundary_id]))
        bbox = np.asarray(boundary_measure.getBoundingBox(), dtype=float)
        if (abs(bbox[4]-z_um) <= tolerance_um and
                abs(bbox[5]-z_um) <= tolerance_um):
            matches.append(boundary_id)
    if not matches:
        raise RuntimeError(
            f'No horizontal faces found at z={z_um} um for domains {domain_ids}.'
        )
    return sorted(matches)

top_oxide_top_faces = horizontal_faces(top_oxide_ids, TOP_METAL_Z_MIN_UM)
top_oxide_bottom_faces = horizontal_faces(
    top_oxide_ids, GROUND_PLANE_Z_MAX_UM
)
bottom_oxide_top_faces = horizontal_faces(
    bottom_oxide_ids, BOTTOM_OXIDE_Z_MAX_UM
)
bottom_oxide_bottom_faces = horizontal_faces(
    bottom_oxide_ids, BOTTOM_OXIDE_Z_MIN_UM
)

# Direct face lists keep hundreds of metal sidewalls out of local Size nodes.
# Only the horizontal faces receive requested electrode refinement.
rf_top_faces = horizontal_faces(rf_domain_ids, TOP_METAL_Z_MAX_UM)
rf_bottom_faces = horizontal_faces(rf_domain_ids, TOP_METAL_Z_MIN_UM)
patterned_gnd_bottom_faces = horizontal_faces(
    top_gnd_domain_ids, TOP_METAL_Z_MIN_UM
)
rf_bottom_on_oxide = sorted(
    set(rf_bottom_faces) & set(top_oxide_top_faces)
)
gnd_bottom_on_oxide = sorted(
    set(patterned_gnd_bottom_faces) & set(top_oxide_top_faces)
)
if not rf_bottom_on_oxide:
    raise RuntimeError('RF has no bottom face on the top oxide.')
if not gnd_bottom_on_oxide:
    raise RuntimeError('Patterned GND has no bottom face on the top oxide.')

air_gap_source_faces = horizontal_faces(
    air_gap_ids, AIR_GAP_Z_MIN_UM
)
air_gap_target_faces = horizontal_faces(
    air_gap_ids, AIR_GAP_Z_MAX_UM
)
air_side_top_source_faces = horizontal_faces(
    air_side_top_ids, TOP_METAL_Z_MIN_UM
)
air_side_top_target_faces = horizontal_faces(
    air_side_top_ids, GROUND_PLANE_Z_MAX_UM
)
air_side_plane_source_faces = horizontal_faces(
    air_side_plane_ids, GROUND_PLANE_Z_MAX_UM
)
air_side_plane_target_faces = horizontal_faces(
    air_side_plane_ids, GROUND_PLANE_Z_MIN_UM
)
air_side_bottom_source_faces = horizontal_faces(
    air_side_bottom_ids, BOTTOM_OXIDE_Z_MAX_UM
)
air_side_bottom_target_faces = horizontal_faces(
    air_side_bottom_ids, BOTTOM_OXIDE_Z_MIN_UM
)
air_side_silicon_source_faces = horizontal_faces(
    air_side_silicon_ids, SILICON_Z_MAX_UM
)
air_side_silicon_target_faces = horizontal_faces(
    air_side_silicon_ids, SILICON_Z_MIN_UM
)
silicon_bottom_faces = horizontal_faces(silicon_ids, SILICON_Z_MIN_UM)
air_lower_source_faces = horizontal_faces(
    air_lower_ids, SILICON_Z_MIN_UM
)
air_lower_target_faces = horizontal_faces(air_lower_ids, AIR_Z_MIN_UM)
expected_lower_source_faces = sorted(
    set(silicon_bottom_faces) | set(air_side_silicon_target_faces)
)
if set(air_lower_source_faces) != set(expected_lower_source_faces):
    raise RuntimeError(
        'Lower-air sweep source is not exactly the silicon-bottom plus '
        'side-air interface.'
    )
# TOP_OXIDE_SOURCE is meshed earlier.  Only the part of the air-gap source
# outside the chip still needs a separate Free Tri feature.
air_gap_unmeshed_source_faces = sorted(
    set(air_gap_source_faces) - set(top_oxide_top_faces)
)
if not air_gap_unmeshed_source_faces:
    raise RuntimeError('No unmeshed exterior part of the air-gap source was found.')
# The layout is about 1 um wider than the specified chip in y, so the
# top-side-air source also contains small electrode-overhang patches. The
# air-gap portion must therefore be a subset, not necessarily an exact match.
if not set(air_gap_unmeshed_source_faces).issubset(
        set(air_side_top_source_faces)):
    raise RuntimeError(
        'The exterior air-gap source is not contained in the side-air source.'
    )
interface_checks = {
    'top-side / plane-side air': (
        air_side_top_target_faces, air_side_plane_source_faces
    ),
    'plane-side / bottom-side air': (
        air_side_plane_target_faces, air_side_bottom_source_faces
    ),
    'bottom-side / silicon-side air': (
        air_side_bottom_target_faces, air_side_silicon_source_faces
    ),
}
for interface_name, (first_faces, second_faces) in interface_checks.items():
    if set(first_faces) != set(second_faces):
        raise RuntimeError(
            f'Nonmatching sweep interface {interface_name}: '
            f'{first_faces} versus {second_faces}'
        )

explicit_selection('TOP_OXIDE_SOURCE', '03 Mesh | Top oxide source', 2,
                   top_oxide_top_faces)
explicit_selection('TOP_OXIDE_TARGET', '03 Mesh | Top oxide target', 2,
                   top_oxide_bottom_faces)
explicit_selection('BOTTOM_OXIDE_SOURCE', '03 Mesh | Bottom oxide source', 2,
                   bottom_oxide_top_faces)
explicit_selection('BOTTOM_OXIDE_TARGET', '03 Mesh | Bottom oxide target', 2,
                   bottom_oxide_bottom_faces)
explicit_selection('AIR_GAP_SOURCE', '03 Mesh | Air-gap sweep source', 2,
                   air_gap_source_faces)
explicit_selection('AIR_GAP_TARGET', '03 Mesh | Air-gap sweep target', 2,
                   air_gap_target_faces)
explicit_selection('AIR_GAP_SOURCE_OUTSIDE',
                   '03 Mesh | Air-gap source outside chip', 2,
                   air_gap_unmeshed_source_faces)
explicit_selection('AIR_SIDE_TOP_SOURCE',
                   '03 Mesh | Full side-air source at z=-0.25 um', 2,
                   air_side_top_source_faces)
explicit_selection('AIR_SIDE_TOP_TARGET',
                   '03 Mesh | Side-top air target', 2,
                   air_side_top_target_faces)
explicit_selection('AIR_SIDE_PLANE_TARGET',
                   '03 Mesh | Side-plane air target', 2,
                   air_side_plane_target_faces)
explicit_selection('AIR_SIDE_BOTTOM_TARGET',
                   '03 Mesh | Side-bottom air target', 2,
                   air_side_bottom_target_faces)
explicit_selection('AIR_SIDE_SILICON_TARGET',
                   '03 Mesh | Side-silicon air target', 2,
                   air_side_silicon_target_faces)
explicit_selection('AIR_LOWER_SOURCE',
                   '03 Mesh | Lower-air source at silicon bottom', 2,
                   air_lower_source_faces)
explicit_selection('AIR_LOWER_TARGET',
                   '03 Mesh | Lower-air target at z=-1000 um', 2,
                   air_lower_target_faces)
print('Top metal domains :', top_metal_domain_ids)
print('RF domain/bbox    :', rf_domain_ids, top_domain_bboxes[rf_domain_ids[0]])
print('Top GND domains   :', top_gnd_domain_ids)
print('Buried GND domain :', ground_plane_domain_ids)
print('RF boundary count :', len(rf_boundary_ids))
print('Patterned GND boundary count:', len(patterned_gnd_boundary_ids))
print('Buried GND boundary count   :', len(buried_gnd_boundary_ids))
print('Combined GND boundary count :', len(gnd_boundary_ids))
print('Air gap/upper domains:', air_gap_ids, air_upper_ids)
print('Air side domains:', {
    'top': air_side_top_ids, 'plane': air_side_plane_ids,
    'bottom': air_side_bottom_ids, 'silicon': air_side_silicon_ids,
    'lower': air_lower_ids,
})
print('All air/SIO2/Si     :', air_ids, sio2_ids, silicon_ids)
print('All conductor faces touch a solved dielectric domain.')
print('Sweep face counts :', {
    'top oxide': (len(top_oxide_top_faces), len(top_oxide_bottom_faces)),
    'bottom oxide': (len(bottom_oxide_top_faces), len(bottom_oxide_bottom_faces)),
    'air gap': (len(air_gap_source_faces), len(air_gap_target_faces)),
    'lower air': (len(air_lower_source_faces), len(air_lower_target_faces)),
})


Top metal domains : [12, 14, 16, 19, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
RF domain/bbox    : [28] [ 9.95112988e+03  1.52760000e+04 -1.70371994e+02  9.46627991e+02
 -2.50000000e-01  2.50000000e-01]
Top GND domains   : [12, 14, 16, 19, 21, 23, 24, 25, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
Buried GND domain : [10]
RF boundary count : 18
Patterned GND boundary count: 714
Buried GND boundary count   : 6
Combined GND boundary count : 720
Air gap/upper domains: [6, 13, 15, 17, 18, 20, 22, 41, 42, 43] [7]
Air side domains: {'top': [5], 'plane': [4], 'bottom': [3], 'silicon': [2], 'lower': [1]}
All air/SIO2/Si     : [1, 2, 3, 4, 5, 6, 7, 13, 15, 17, 18, 20, 22, 41, 42, 43] [9, 11] [8]
All conductor faces touch a solved dielectric domain.
Sweep face counts : {'top oxide': (33, 1), 'bottom oxide': (1, 1), 'air gap': (11, 10), 'lower air': (2, 1)}


## 5. Materials and electrostatics

Air, both oxide layers, and silicon are solved as dielectric domains. Relative permittivities are 1.0, 3.9, and 11.7. All conductor interiors are excluded from Electrostatics. RF is held at 1 V; the other patterned electrodes and the complete buried plane are grounded.

In [6]:
def add_isotropic_material(tag, label, epsr, selection_tag):
    material = comp.material().create(tag, 'Common')
    material.label(label)
    material.selection().named(selection_tag)
    tensor = [str(epsr), '0', '0', '0', str(epsr), '0', '0', '0', str(epsr)]
    material.propertyGroup('def').set('relpermittivity', tensor)
    return material

add_isotropic_material('mat_air', 'Air', 1.0, 'AIR')
add_isotropic_material('mat_sio2', 'SiO2', 3.9, 'SIO2')
add_isotropic_material('mat_silicon', 'Silicon (dielectric)', 11.7, 'SILICON')

j.param().set('Vrf', '1[V]', 'RF terminal voltage')
comp.physics().create('es', 'Electrostatics', 'geom1')
es = comp.physics('es')
es.selection().named('DIELECTRICS')

es_tags = [str(tag) for tag in es.feature().tags()]
if 'ccn1' not in es_tags:
    raise RuntimeError(f'Default Charge Conservation is missing: {es_tags}')
es.feature('ccn1').label('Charge Conservation: all dielectric domains')
if 'zerochg1' in es_tags:
    es.feature('zerochg1').label('Zero charge on exterior air boundaries')

es.create('gnd1', 'Ground', JInt(2))
es.feature('gnd1').label('GND: top DC electrodes + buried plane')
es.feature('gnd1').selection().named('GND')
es.create('term1', 'Terminal', JInt(2))
es.feature('term1').label('RF terminal: 1 V')
es.feature('term1').selection().named('RF')
es.feature('term1').set('TerminalType', 'Voltage')
es.feature('term1').set('V0', 'Vrf')
print('Electrostatics configured; no solve has been run.')


Electrostatics configured; no solve has been run.


## 6. Local mesh, stationary study, and capacitance checks

This is a **fine convergence mesh**. Its maximum/minimum sizes are RF 20/0.2 um, patterned GND 50/0.2 um, buried GND 150/4 um, top SiO2 50/0.5 um, bottom SiO2 200/4 um, silicon 400/20 um, near-electrode air 50/0.2 um, side air 200/4 um, upper air 600/0.5 um, and lower air 800/20 um. Refinement is concentrated near the active electrodes, 3 um top oxide, and upper fringing field. The already-proven settings in the shielded lower stack are deliberately retained to reduce the risk of another meshing stall.

The air encloses the complete chip. The near-electrode air uses three prism layers; the 3 um top oxide uses five. Rectangular air rings are swept beside the 3 um oxide, 0.5 um buried plane, 12 um oxide, and 675 um silicon with 5/1/4/6 layers, while the 12 um bottom oxide uses four. The lower-air block is swept through three prism layers from the already-meshed silicon-bottom/side-air interface to z = -1000 um. Consequently, no tetrahedral volume has to follow a 0.5 um metal sidewall or reconcile the large, partitioned lower interface. Only the upper bulk-air block and silicon use free tetrahedra. Metal volumes remain outside every volume-mesh feature.

The enclosure has 250 um lateral padding, reaches 250 um above the electrodes, and reaches z = -1000 um below the chip. Every conductor boundary is checked for adjacency to a solved dielectric, preventing the previous `No mesh on boundaries ...` warning caused by exposed conductor faces. Before reporting a final capacitance, enlarge the exterior-air distances and refine the RF/top-oxide mesh until both capacitance evaluations stop changing appreciably.

A dedicated Solution dataset, `dsetcap`, uses COMSOL's first compatible solution, which is created when Study 1 is computed. Global Evaluation writes to the named table `tblcap`. After computing Study 1, select **Capacitance and charge-conservation checks** and click **Evaluate**; its Dataset field should show **Study 1 solution for capacitance**, not **None**. Section 6 does not generate a solver sequence.

This cell creates the mesh sequence and study but deliberately calls neither `mesh.run()` nor `study.run()`.

In [7]:
comp.mesh().create('mesh1')
mesh = comp.mesh('mesh1')
mesh.label('Fine convergence mesh | swept oxides + tetrahedral bulk')

def add_mesh_size(parent, tag, label, settings, selection_tag=None,
                  entity_ids=None, entity_dimension=None):
    if (selection_tag is None) == (entity_ids is None):
        raise ValueError('Provide exactly one of selection_tag or entity_ids.')
    parent.create(tag, 'Size')
    size = parent.feature(tag)
    size.label(label)
    if selection_tag is not None:
        size.selection().named(selection_tag)
    else:
        if entity_dimension is None:
            raise ValueError('entity_dimension is required with entity_ids.')
        size.selection().geom('geom1', JInt(entity_dimension))
        size.selection().set(JArray(JInt)(list(entity_ids)))
    size.set('custom', JBoolean(True))
    size.set('hmax', f"{settings['hmax']}[um]")
    size.set('hmin', f"{settings['hmin']}[um]")
    size.set('hgrad', str(settings['growth']))
    return size

def add_swept_domain(tag, label, domain_selection, source_selection,
                     target_selection, settings, layers):
    mesh.create(tag, 'Sweep')
    sweep = mesh.feature(tag)
    sweep.label(label)
    sweep.selection().named(domain_selection)
    sweep.selection('sourceface').named(source_selection)
    sweep.selection('targetface').named(target_selection)
    sweep.set('facemethod', 'tri')  # triangular faces -> prism elements
    sweep.set('sweeppath', 'straight')
    add_mesh_size(
        sweep, 'size1', f'{label}: lateral size', settings,
        selection_tag=domain_selection
    )
    sweep.create('dist1', 'Distribution')
    distribution = sweep.feature('dist1')
    distribution.label(f'{layers} elements through thickness')
    distribution.selection().named(domain_selection)
    distribution.set('numelem', JInt(layers))
    return sweep

# 01-02: mesh the partitioned top-oxide surface and sweep through 3 um.
# Local electrode sizes are restricted to horizontal faces.
mesh.create('ftri_top', 'FreeTri')
ftri_top = mesh.feature('ftri_top')
ftri_top.label('01 | Triangular source on top oxide')
ftri_top.selection().named('TOP_OXIDE_SOURCE')
add_mesh_size(
    ftri_top, 'size_sio2', 'Top SiO2: 50/0.5 um', MESH_SIO2_TOP,
    selection_tag='TOP_OXIDE_SOURCE'
)
add_mesh_size(
    ftri_top, 'size_gnd', 'Patterned GND bottom faces: 50/0.2 um',
    MESH_PATTERNED_GND, entity_ids=gnd_bottom_on_oxide, entity_dimension=2
)
add_mesh_size(
    ftri_top, 'size_rf', 'RF bottom face: 20/0.2 um', MESH_RF,
    entity_ids=rf_bottom_on_oxide, entity_dimension=2
)
add_swept_domain(
    'swe_top_oxide', '02 | Sweep 3 um top SiO2', 'SIO2_TOP',
    'TOP_OXIDE_SOURCE', 'TOP_OXIDE_TARGET', MESH_SIO2_TOP, 5
)

# 03-08: start from the complete side-air source at z=-0.25 um. It includes
# the simple exterior ring plus any small electrode-overhang patches. The
# same conforming mesh feeds the upper gap and every downward side-air sweep.
mesh.create('ftri_air_gap', 'FreeTri')
ftri_air_gap = mesh.feature('ftri_air_gap')
ftri_air_gap.label('03 | Triangular source outside chip')
ftri_air_gap.selection().named('AIR_SIDE_TOP_SOURCE')
add_mesh_size(
    ftri_air_gap, 'size_air_side', 'Side air source: 200/4 um',
    MESH_AIR_SIDE, selection_tag='AIR_SIDE_TOP_SOURCE'
)
add_swept_domain(
    'swe_air_gap', '04 | Sweep 0.5 um near-electrode air', 'AIR_GAP',
    'AIR_GAP_SOURCE', 'AIR_GAP_TARGET', MESH_AIR_GAP, 3
)
add_swept_domain(
    'swe_air_side_top', '05 | Sweep side air beside top SiO2',
    'AIR_SIDE_TOP', 'AIR_SIDE_TOP_SOURCE', 'AIR_SIDE_TOP_TARGET',
    MESH_AIR_SIDE, 5
)
add_swept_domain(
    'swe_air_side_plane', '06 | Sweep side air beside buried GND',
    'AIR_SIDE_PLANE', 'AIR_SIDE_TOP_TARGET', 'AIR_SIDE_PLANE_TARGET',
    MESH_AIR_SIDE, 1
)

# 07-08: mesh the 12 um oxide source only after the side-air sweep has
# established its perimeter mesh, then sweep both adjoining regions with
# the same three through-thickness layers.
mesh.create('ftri_buried', 'FreeTri')
ftri_buried = mesh.feature('ftri_buried')
ftri_buried.label('07 | Triangular source below buried GND')
ftri_buried.selection().named('BOTTOM_OXIDE_SOURCE')
add_mesh_size(
    ftri_buried, 'size_sio2', 'Bottom SiO2: 200/4 um',
    MESH_SIO2_BOTTOM, selection_tag='BOTTOM_OXIDE_SOURCE'
)
add_mesh_size(
    ftri_buried, 'size_plane', 'Buried GND bottom face: 150/4 um',
    MESH_BURIED_GND, entity_ids=bottom_oxide_top_faces, entity_dimension=2
)
add_swept_domain(
    'swe_bottom_oxide', '08 | Sweep 12 um bottom SiO2', 'SIO2_BOTTOM',
    'BOTTOM_OXIDE_SOURCE', 'BOTTOM_OXIDE_TARGET', MESH_SIO2_BOTTOM, 4
)
add_swept_domain(
    'swe_air_side_bottom', '09 | Sweep side air beside bottom SiO2',
    'AIR_SIDE_BOTTOM', 'AIR_SIDE_PLANE_TARGET',
    'AIR_SIDE_BOTTOM_TARGET', MESH_AIR_SIDE, 4
)
add_swept_domain(
    'swe_air_side_silicon', '10 | Sweep side air beside silicon',
    'AIR_SIDE_SILICON', 'AIR_SIDE_BOTTOM_TARGET',
    'AIR_SIDE_SILICON_TARGET', MESH_AIR_SIDE, 6
)

# 11: the side-air sweep meshes the silicon sidewalls first; the silicon
# tetrahedra then conform to those faces.
mesh.create('ftet_silicon', 'FreeTet')
ftet_silicon = mesh.feature('ftet_silicon')
ftet_silicon.label('11 | Free tetrahedra in silicon')
ftet_silicon.selection().named('SILICON')
add_mesh_size(
    ftet_silicon, 'size_si', 'Silicon: 400/20 um', MESH_SILICON,
    selection_tag='SILICON'
)

# 12: reuse the fully meshed silicon-bottom/side-air interface and sweep it
# through the simple lower box. This avoids the expensive tetrahedral
# 'Respecting boundaries' stage that previously stalled with zero elements.
add_swept_domain(
    'swe_air_lower', '12 | Sweep lower air', 'AIR_LOWER',
    'AIR_LOWER_SOURCE', 'AIR_LOWER_TARGET', MESH_AIR_LOWER, 3
)

# 13: the upper air remains a geometrically simple tetrahedral volume.
mesh.create('ftet_air_upper', 'FreeTet')
ftet_air_upper = mesh.feature('ftet_air_upper')
ftet_air_upper.label('13 | Free tetrahedra in upper air')
ftet_air_upper.selection().named('AIR_UPPER')
add_mesh_size(
    ftet_air_upper, 'size_air_upper', 'Upper air: 600/0.5 um',
    MESH_AIR_UPPER, selection_tag='AIR_UPPER'
)
add_mesh_size(
    ftet_air_upper, 'size_rf_top', 'RF top face in air: 50/0.5 um',
    MESH_RF_AIR,
    entity_ids=rf_top_faces, entity_dimension=2
)

j.study().create('std1')
j.study('std1').label('Stationary capacitance study')
j.study('std1').create('stat', 'Stationary')

# Component integration operators can be configured before a solution exists.
comp.cpl().create('intWop', 'Integration', 'geom1')
comp.cpl('intWop').label('Electrostatic energy over all dielectrics')
comp.cpl('intWop').set('opname', 'intWop')
comp.cpl('intWop').selection().named('DIELECTRICS')

comp.cpl().create('intQrfop', 'Integration', 'geom1')
comp.cpl('intQrfop').label('RF free charge')
comp.cpl('intQrfop').set('opname', 'intQrfop')
comp.cpl('intQrfop').selection().named('RF')

comp.cpl().create('intQgndop', 'Integration', 'geom1')
comp.cpl('intQgndop').label('Total GND free charge')
comp.cpl('intQgndop').set('opname', 'intQgndop')
comp.cpl('intQgndop').selection().named('GND')

j.result().dataset().create('dsetcap', 'Solution')
dset_cap = j.result().dataset('dsetcap')
dset_cap.label('Study 1 solution for capacitance')
# Do not create or name a solver here. With one study, the Solution dataset
# automatically uses the first compatible solution after Study 1 computes.
dset_cap.set('comp', 'comp1')

j.result().table().create('tblcap', 'Table')
cap_table = j.result().table('tblcap')
cap_table.label('Capacitance and charge checks')

j.result().numerical().create('gevCap', 'EvalGlobal')
gev = j.result().numerical('gevCap')
gev.label('Capacitance and charge-conservation checks')
gev.set('data', 'dsetcap')
gev.set('table', 'tblcap')
gev.set('expr', [
    '2*intWop(es.We)/Vrf^2',
    'abs(intQrfop(es.nD))/abs(Vrf)',
    'intQrfop(es.nD)',
    'intQgndop(es.nD)',
    'abs(intQrfop(es.nD)+intQgndop(es.nD))/'
    'max(abs(intQrfop(es.nD)),1e-30[C])',
])
gev.set('unit', ['F', 'F', 'C', 'C', '1'])
gev.set('descr', [
    'Capacitance from stored energy',
    'Capacitance from RF charge',
    'RF free charge',
    'Total GND free charge',
    'Relative conductor charge imbalance',
])

print('Fine convergence mesh controls:')
print('  RF            ', MESH_RF)
print('  Patterned GND ', MESH_PATTERNED_GND)
print('  Buried GND    ', MESH_BURIED_GND)
print('  Top SiO2      ', MESH_SIO2_TOP)
print('  Bottom SiO2   ', MESH_SIO2_BOTTOM)
print('  Silicon       ', MESH_SILICON)
print('  Air gap       ', MESH_AIR_GAP)
print('  Side air      ', MESH_AIR_SIDE)
print('  Upper air     ', MESH_AIR_UPPER)
print('  Lower air     ', MESH_AIR_LOWER)
print('  RF top/air    ', MESH_RF_AIR)
print('  Swept layers: top oxide=5, air gap=3, side air=5/1/4/6,')
print('                bottom oxide=4, lower air=3')
print('  Free tetrahedra: silicon and upper air only')
print('  Air enclosure [um]: xy padding=250, z=-1000 to +250')
print('  Results: dataset=dsetcap, table=tblcap')
print('Mesh and stationary study configured but NOT run.')


Fine convergence mesh controls:
  RF             {'hmax': 20.0, 'hmin': 0.2, 'growth': 1.45}
  Patterned GND  {'hmax': 50.0, 'hmin': 0.2, 'growth': 1.6}
  Buried GND     {'hmax': 150.0, 'hmin': 4.0, 'growth': 1.7}
  Top SiO2       {'hmax': 50.0, 'hmin': 0.5, 'growth': 1.5}
  Bottom SiO2    {'hmax': 200.0, 'hmin': 4.0, 'growth': 1.7}
  Silicon        {'hmax': 400.0, 'hmin': 20.0, 'growth': 1.75}
  Air gap        {'hmax': 50.0, 'hmin': 0.2, 'growth': 1.6}
  Side air       {'hmax': 200.0, 'hmin': 4.0, 'growth': 1.75}
  Upper air      {'hmax': 600.0, 'hmin': 0.5, 'growth': 1.8}
  Lower air      {'hmax': 800.0, 'hmin': 20.0, 'growth': 1.85}
  RF top/air     {'hmax': 50.0, 'hmin': 0.5, 'growth': 1.7}
  Swept layers: top oxide=5, air gap=3, side air=5/1/4/6,
                bottom oxide=4, lower air=3
  Free tetrahedra: silicon and upper air only
  Air enclosure [um]: xy padding=250, z=-1000 to +250
  Results: dataset=dsetcap, table=tblcap
Mesh and stationary study configured but NOT run.


In [8]:
repo_output_dir = Path.cwd() / 'Simulations' / 'trap_capacitance'
output_dir = repo_output_dir if repo_output_dir.is_dir() else Path.cwd()
setup_file = output_dir / 'COMSOL_MODEL_V3_buried_GND_capacitance_setup.mph'
model.save(str(setup_file))

## 7. Build and inspect the mesh only

The code in this cell remains commented so it cannot lock the notebook accidentally. Either build **Mesh 1** in the COMSOL GUI after running the save cell, or uncomment this cell to build only the mesh without starting the stationary solver. The diagnostic reports element counts, minimum quality, and mesh problem nodes.

In [9]:
# # Mesh-only test: this does not run Study 1.
# mesh.run()

# mesh_types = [str(value) for value in mesh.getTypes()]
# print('Mesh built successfully.')
# print('Vertices      :', int(mesh.getNumVertex()))
# print('Total elements:', int(mesh.getNumElem()))
# print('Element counts:')
# for element_type in mesh_types:
#     count = int(mesh.getNumElem(element_type))
#     if count:
#         print(f'  {element_type:8s}: {count}')
# try:
#     print('Minimum quality:', float(mesh.getMinQuality()))
# except Exception as exc:
#     print('Minimum quality was not available:', exc)
# print('Mesh problem nodes:', [str(value) for value in mesh.problems()])


## 8. Save the unsolved setup

Run this cell to create the new `.mph`. Open it in COMSOL and select **Study 1 > Compute**; COMSOL will build Mesh 1 automatically. Then evaluate **Results > Derived Values > Capacitance and charge-conservation checks**. Compare the energy- and charge-derived capacitances and perform a mesh-refinement study before treating the result as converged.

In [10]:
# repo_output_dir = Path.cwd() / 'Simulations' / 'trap_capacitance'
# output_dir = repo_output_dir if repo_output_dir.is_dir() else Path.cwd()
# setup_file = output_dir / 'COMSOL_MODEL_V3_buried_GND_capacitance_setup.mph'
# model.save(str(setup_file))
# print('Saved configured, unsolved model:', setup_file.resolve())
# print('Next: open it in COMSOL and run Study 1. No solve was run here.')
